
# Fixed Config Evaluation

Fit selected GP and KNN configurations on the full dataset for prediction surfaces, then evaluate the same fixed configurations on outer geographic splits. Robustness MSE rows are saved under `../eval_results/fixed_config/` and skipped on rerun unless `FORCE_REFIT=True`.


In [ ]:

import os
import sys
import tempfile
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import dill
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

CWD = Path.cwd()
if CWD.name == "gridsearch":
    NOTEBOOK_DIR = CWD
elif (CWD / "gridsearch").exists() and CWD.name == "code":
    NOTEBOOK_DIR = CWD / "gridsearch"
elif (CWD / "code" / "gridsearch").exists():
    NOTEBOOK_DIR = CWD / "code" / "gridsearch"
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(CODE_DIR))

from gp import GaussianProcess
from gp_kernels import make_indoor_outdoor_mean, make_wifi_kernel
from knn_features import WifiKNNFeatureTransformer
from eval_workflow import completed_gp_results, summarize_cv_results, write_result_row
import wifiplotting


In [ ]:

DATA_DIR = CODE_DIR / "data"
GEO_CV_DIR = CODE_DIR / "cv" / "geo"
EVAL_RESULTS_DIR = CODE_DIR / "eval_results"
FIXED_RESULT_DIR = EVAL_RESULTS_DIR / "fixed_config"
FIXED_RESULT_DIR.mkdir(parents=True, exist_ok=True)

GP_FINAL_RESULT_PATH = EVAL_RESULTS_DIR / "gp_final_holdout.csv"
KNN_FINAL_RESULT_PATH = EVAL_RESULTS_DIR / "knn_final_holdout.csv"
GP_STAGE1_RESULT_DIR = EVAL_RESULTS_DIR / "gp_stage1"
GP_STAGE2_RESULT_DIR = EVAL_RESULTS_DIR / "gp_stage2"
GP_GRID_PATH = CODE_DIR / "gridsearch" / "gp_geo_stage1_grid.csv"

ROBUSTNESS_FOLDS = [1, 2, 3, 4]
FORCE_REFIT = False
GEO_VIS_SPLIT_IDX = 2
GEO_VIS_LAT_BINS = 20
GEO_VIS_LON_BINS = 20

N_CHAINS = 2
N_SAMPLES = 111
BURNIN = 10
THIN = 10
CALIBRATION_ITERS = 30
JITTER_FACTOR = 5
PREDICT_METHOD = "sequential"
KEY_SEED = 305


In [ ]:

GP_PARAM_COLS = ["ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap"]
KNN_PARAM_COLS = [
    "features__indoor_scale",
    "features__ap_scale",
    "knn__n_neighbors",
    "knn__weights",
    "knn__p",
]


def _as_plain_config(row, cols):
    config = {}
    for col in cols:
        value = row[col]
        if isinstance(value, np.generic):
            value = value.item()
        config[col] = value
    return config


def load_default_gp_config():
    fallback = {
        "ap_form": "add",
        "ls_xy": 0.04,
        "ls_z": 0.5,
        "os_xyz": 25.0,
        "ls_t": 50.0,
        "os_t": 5.0,
        "ls_ap": 0.5,
        "os_ap": 0.5,
    }

    if GP_FINAL_RESULT_PATH.exists():
        row = pd.read_csv(GP_FINAL_RESULT_PATH).iloc[0]
        return _as_plain_config(row, GP_PARAM_COLS)

    grid = pd.read_csv(GP_GRID_PATH) if GP_GRID_PATH.exists() else None
    for result_dir, n_folds in [(GP_STAGE2_RESULT_DIR, 5), (GP_STAGE1_RESULT_DIR, 3)]:
        results = completed_gp_results(result_dir)
        if results.empty:
            continue
        summary = summarize_cv_results(results)
        complete = summary[summary["n_folds"] == n_folds]
        if complete.empty:
            continue
        best = complete.iloc[0]
        if grid is not None:
            best = best.to_frame().T.merge(grid, on="param_id", how="left").iloc[0]
        return _as_plain_config(best, GP_PARAM_COLS)

    return fallback


def load_default_knn_config():
    fallback = {
        "features__indoor_scale": 0.5,
        "features__ap_scale": 1.0,
        "knn__n_neighbors": 15,
        "knn__weights": "distance",
        "knn__p": 1,
    }
    if KNN_FINAL_RESULT_PATH.exists():
        row = pd.read_csv(KNN_FINAL_RESULT_PATH).iloc[0]
        return _as_plain_config(row, KNN_PARAM_COLS)
    return fallback


GP_CONFIG = load_default_gp_config()
KNN_CONFIG = load_default_knn_config()

# Edit these dictionaries directly before running expensive cells if needed.
GP_CONFIG, KNN_CONFIG


In [ ]:

# Optional manual overrides. Leave empty to use the defaults above.
GP_CONFIG.update({})
KNN_CONFIG.update({})

KNN_CONFIG["knn__n_neighbors"] = int(KNN_CONFIG["knn__n_neighbors"])
KNN_CONFIG["knn__p"] = int(KNN_CONFIG["knn__p"])
GP_CONFIG, KNN_CONFIG


In [ ]:

def load_full_data():
    return {
        "X": np.load(DATA_DIR / "X_train.npy"),
        "y": np.load(DATA_DIR / "y_train.npy"),
        "obs_count": np.load(DATA_DIR / "obs_count.npy"),
        "obs_sse": np.load(DATA_DIR / "obs_sse.npy"),
        "coord": np.load(DATA_DIR / "coord_train.npy"),
        "world": np.load(DATA_DIR / "world_train.npy"),
    }


def load_prediction_grid():
    return {
        "X": np.load(DATA_DIR / "X_test.npy"),
        "coord": np.load(DATA_DIR / "coord_test.npy"),
        "world": np.load(DATA_DIR / "world_test.npy"),
    }


def load_geo_fold(fold_idx):
    fold_idx = int(fold_idx)
    return {
        "X_train": np.load(GEO_CV_DIR / f"X_train_{fold_idx}.npy"),
        "y_train": np.load(GEO_CV_DIR / f"y_train_{fold_idx}.npy"),
        "obs_count_train": np.load(GEO_CV_DIR / f"obs_count_train_{fold_idx}.npy"),
        "obs_sse_train": np.load(GEO_CV_DIR / f"obs_sse_train_{fold_idx}.npy"),
        "world_train": np.load(GEO_CV_DIR / f"world_train_{fold_idx}.npy"),
        "train_idx": np.load(GEO_CV_DIR / f"train_idx_{fold_idx}.npy"),
        "X_test": np.load(GEO_CV_DIR / f"X_test_{fold_idx}.npy"),
        "y_test": np.load(GEO_CV_DIR / f"y_test_{fold_idx}.npy"),
        "world_test": np.load(GEO_CV_DIR / f"world_test_{fold_idx}.npy"),
        "test_idx": np.load(GEO_CV_DIR / f"test_idx_{fold_idx}.npy"),
    }


def fixed_result_path(model, fold_idx):
    return FIXED_RESULT_DIR / f"geo_{model}_fold_{int(fold_idx)}.csv"


In [ ]:

def fit_gp(X_train, y_train, obs_count_train, obs_sse_train, config):
    X_train = jnp.asarray(X_train)
    y_train = jnp.asarray(y_train)
    m = make_indoor_outdoor_mean(X_train, y_train)
    K = make_wifi_kernel(**config)
    gp = GaussianProcess(m, K)
    gp.fit(
        X_train,
        y_train,
        jnp.asarray(obs_count_train),
        obs_sse=jnp.asarray(obs_sse_train),
    )
    return gp


def sample_gp(gp, *, key):
    start = time.time()
    chain = gp.gibbs(
        key=key,
        chains=N_CHAINS,
        samples=N_SAMPLES,
        calibration_iters=CALIBRATION_ITERS,
        jitter_factor=JITTER_FACTOR,
    )
    return chain, time.time() - start


def predict_gp_from_chain(gp, X_pred, chain):
    cov_chains = chain[1][:, BURNIN::THIN, :]
    start = time.time()
    pred_means, pred_vars = gp.predict(jnp.asarray(X_pred), cov_chains, method=PREDICT_METHOD)
    return np.asarray(pred_means.mean(axis=0)), time.time() - start


def run_gp_predictions(gp, X_pred, *, key):
    chain, fit_elapsed = sample_gp(gp, key=key)
    y_pred, predict_elapsed = predict_gp_from_chain(gp, X_pred, chain)
    return y_pred, chain, fit_elapsed, predict_elapsed


def make_knn_model(config, X_train):
    feature_config = {
        "indoor_scale": float(config["features__indoor_scale"]),
        "ap_scale": float(config["features__ap_scale"]),
        "n_ap": int(np.nanmax(X_train[:, 4])) + 1,
    }
    knn_config = {
        "n_neighbors": int(config["knn__n_neighbors"]),
        "weights": config["knn__weights"],
        "p": int(config["knn__p"]),
    }
    return Pipeline([
        ("features", WifiKNNFeatureTransformer(**feature_config)),
        ("knn", KNeighborsRegressor(**knn_config)),
    ])


def fit_knn(X_train, y_train, config):
    model = make_knn_model(config, X_train)
    model.fit(X_train, y_train)
    return model



## Full-Data Prediction Surfaces

These cells fit on all `code/data` training rows and predict over the grid in `code/data/X_test.npy`.


In [ ]:

full_data = load_full_data()
pred_grid = load_prediction_grid()
with open(DATA_DIR / "osm_context.pkl", "rb") as f:
    osm_context = dill.load(f)

vmin = np.quantile(full_data["y"], 0.025)
vmax = np.quantile(full_data["y"], 0.975)


In [ ]:

# Expensive: full-data GP fit and grid prediction.
full_gp = fit_gp(
    full_data["X"],
    full_data["y"],
    full_data["obs_count"],
    full_data["obs_sse"],
    GP_CONFIG,
)
full_gp_grid_pred, full_gp_chain, full_gp_fit_elapsed, full_gp_predict_elapsed = run_gp_predictions(
    full_gp,
    pred_grid["X"],
    key=jr.PRNGKey(KEY_SEED + 50_000),
)
full_gp_fit_elapsed, full_gp_predict_elapsed


In [ ]:

fig, ax, surface = wifiplotting.plot_geo_fold_surface(
    osm_context,
    pred_grid["world"],
    full_gp_grid_pred,
    world_train=full_data["world"],
    y_train=full_data["y"],
    vmin=vmin,
    vmax=vmax,
    model_label="GP",
    title="GP fixed-config surface fit on all observations",
    surface_size=10,
    surface_alpha=0.8,
    show_train=True,
    train_size=10,
)


In [ ]:

full_knn = fit_knn(full_data["X"], full_data["y"], KNN_CONFIG)
full_knn_grid_pred = full_knn.predict(pred_grid["X"])

fig, ax, surface = wifiplotting.plot_geo_fold_surface(
    osm_context,
    pred_grid["world"],
    full_knn_grid_pred,
    world_train=full_data["world"],
    y_train=full_data["y"],
    vmin=vmin,
    vmax=vmax,
    model_label="KNN",
    title="KNN fixed-config surface fit on all observations",
    surface_size=36,
    surface_alpha=0.7,
    show_train=True,
    train_size=10,
)



## Outer Geographic Split Robustness

By default this evaluates folds `1-4`; fold `0` is reserved for the tuned outer split workflow.


In [ ]:

def evaluate_gp_fold(fold_idx):
    path = fixed_result_path("gp", fold_idx)
    if path.exists() and not FORCE_REFIT:
        return pd.read_csv(path).iloc[0].to_dict()

    fold = load_geo_fold(fold_idx)
    gp = fit_gp(
        fold["X_train"],
        fold["y_train"],
        fold["obs_count_train"],
        fold["obs_sse_train"],
        GP_CONFIG,
    )
    y_pred, chain, fit_elapsed, predict_elapsed = run_gp_predictions(
        gp,
        fold["X_test"],
        key=jr.PRNGKey(KEY_SEED + 60_000 + int(fold_idx)),
    )
    mse = float(np.mean((fold["y_test"] - y_pred) ** 2))
    row = {
        "model": "gp",
        "fold_idx": int(fold_idx),
        "mse": mse,
        "n_train": int(fold["X_train"].shape[0]),
        "n_test": int(fold["X_test"].shape[0]),
        "fit_elapsed": float(fit_elapsed),
        "predict_elapsed": float(predict_elapsed),
        "jitter": float(gp.jitter),
        **GP_CONFIG,
    }
    write_result_row(path, row)
    return row


def evaluate_knn_fold(fold_idx):
    path = fixed_result_path("knn", fold_idx)
    if path.exists() and not FORCE_REFIT:
        return pd.read_csv(path).iloc[0].to_dict()

    fold = load_geo_fold(fold_idx)
    start = time.time()
    model = fit_knn(fold["X_train"], fold["y_train"], KNN_CONFIG)
    fit_elapsed = time.time() - start
    start = time.time()
    y_pred = model.predict(fold["X_test"])
    predict_elapsed = time.time() - start
    mse = float(np.mean((fold["y_test"] - y_pred) ** 2))
    row = {
        "model": "knn",
        "fold_idx": int(fold_idx),
        "mse": mse,
        "n_train": int(fold["X_train"].shape[0]),
        "n_test": int(fold["X_test"].shape[0]),
        "fit_elapsed": float(fit_elapsed),
        "predict_elapsed": float(predict_elapsed),
        **KNN_CONFIG,
    }
    write_result_row(path, row)
    return row


In [ ]:

# Expensive for GP. Restartable: completed fold CSVs are reused unless FORCE_REFIT=True.
robustness_rows = []
for fold_idx in ROBUSTNESS_FOLDS:
    print(f"Evaluating GP fold {fold_idx}", flush=True)
    robustness_rows.append(evaluate_gp_fold(fold_idx))
    print(f"Evaluating KNN fold {fold_idx}", flush=True)
    robustness_rows.append(evaluate_knn_fold(fold_idx))

robustness_results = pd.DataFrame(robustness_rows)
robustness_results.sort_values(["fold_idx", "model"])


In [ ]:

if "robustness_results" in globals() and not robustness_results.empty:
    robustness_summary = (
        robustness_results
        .groupby("model", as_index=False)
        .agg(mean_mse=("mse", "mean"), std_mse=("mse", "std"), n_folds=("fold_idx", "nunique"))
        .sort_values("mean_mse")
    )
else:
    robustness_summary = pd.DataFrame()
robustness_summary



## One-Fold Surface And Error Diagnostics

Default fold `2` is useful for inspecting the same geographically funky region as prior notebooks. These cells refit the selected fixed configs on that fold so predictions are available for plotting.


In [ ]:

def geo_fold_plot_metadata(fold_idx):
    coord_all = np.load(DATA_DIR / "coord_train.npy")
    y_all = np.load(DATA_DIR / "y_train.npy")
    world_grid = np.load(DATA_DIR / "world_test.npy")
    test_idx = np.load(GEO_CV_DIR / f"test_idx_{fold_idx}.npy")

    lat_edges, lon_edges, lat_grid, lon_grid, in_grid = wifiplotting.geo_vis_grid_from_lonlat(
        coord_all[:, 0],
        coord_all[:, 1],
        GEO_VIS_LAT_BINS,
        GEO_VIS_LON_BINS,
    )
    heldout_cells = wifiplotting.geo_vis_cells_from_indices(test_idx, lat_grid, lon_grid, in_grid)
    return {
        "coord_all": coord_all,
        "y_all": y_all,
        "world_grid": world_grid,
        "test_idx": test_idx,
        "lat_edges": lat_edges,
        "lon_edges": lon_edges,
        "heldout_cells": heldout_cells,
        "vmin": np.quantile(y_all, 0.025),
        "vmax": np.quantile(y_all, 0.975),
    }


def plot_fold_surface_and_errors(model_label, fold, meta, grid_pred, test_pred, *, surface_size):
    fig, ax, surface = wifiplotting.plot_geo_fold_surface(
        osm_context,
        meta["world_grid"],
        grid_pred,
        world_train=fold["world_train"],
        y_train=fold["y_train"],
        world_test=fold["world_test"],
        y_test=fold["y_test"],
        lat_edges=meta["lat_edges"],
        lon_edges=meta["lon_edges"],
        heldout_cells=meta["heldout_cells"],
        vmin=meta["vmin"],
        vmax=meta["vmax"],
        model_label=model_label,
        split_idx=GEO_VIS_SPLIT_IDX,
        surface_size=surface_size,
        surface_alpha=0.75,
        show_train=False,
        test_size=72,
        cell_style={"edgecolor": "black"},
    )

    resid, sq_err, mse, top_local_idx, top_errors = wifiplotting.compute_heldout_error_summary(
        fold["y_test"],
        test_pred,
        test_idx=meta["test_idx"],
        coord_all=meta["coord_all"],
        X_test=fold["X_test"],
        top_n=4,
    )
    fig, ax = wifiplotting.plot_heldout_error_hist(
        sq_err,
        mse,
        top_local_idx,
        split_idx=GEO_VIS_SPLIT_IDX,
        model_label=model_label,
        top_n=len(top_local_idx),
    )
    fig, ax, top_scatter = wifiplotting.plot_top_heldout_errors_map(
        osm_context,
        fold["world_test"],
        sq_err,
        top_local_idx,
        lat_edges=meta["lat_edges"],
        lon_edges=meta["lon_edges"],
        heldout_cells=meta["heldout_cells"],
        split_idx=GEO_VIS_SPLIT_IDX,
        model_label=model_label,
        top_n=len(top_local_idx),
        cell_style={"edgecolor": "black"},
    )
    return mse, top_errors


In [ ]:

# Expensive for GP. Run after choosing GEO_VIS_SPLIT_IDX.
fold = load_geo_fold(GEO_VIS_SPLIT_IDX)
meta = geo_fold_plot_metadata(GEO_VIS_SPLIT_IDX)

vis_gp = fit_gp(
    fold["X_train"],
    fold["y_train"],
    fold["obs_count_train"],
    fold["obs_sse_train"],
    GP_CONFIG,
)
vis_gp_chain, vis_gp_fit_elapsed = sample_gp(
    vis_gp,
    key=jr.PRNGKey(KEY_SEED + 70_000 + GEO_VIS_SPLIT_IDX),
)
vis_gp_grid_pred, vis_gp_grid_predict_elapsed = predict_gp_from_chain(
    vis_gp,
    pred_grid["X"],
    vis_gp_chain,
)
vis_gp_test_pred, vis_gp_test_predict_elapsed = predict_gp_from_chain(
    vis_gp,
    fold["X_test"],
    vis_gp_chain,
)

gp_fold_mse, gp_top_errors = plot_fold_surface_and_errors(
    "GP",
    fold,
    meta,
    vis_gp_grid_pred,
    vis_gp_test_pred,
    surface_size=10,
)
gp_fold_mse, gp_top_errors


In [ ]:

vis_knn = fit_knn(fold["X_train"], fold["y_train"], KNN_CONFIG)
vis_knn_grid_pred = vis_knn.predict(pred_grid["X"])
vis_knn_test_pred = vis_knn.predict(fold["X_test"])

knn_fold_mse, knn_top_errors = plot_fold_surface_and_errors(
    "KNN",
    fold,
    meta,
    vis_knn_grid_pred,
    vis_knn_test_pred,
    surface_size=36,
)
knn_fold_mse, knn_top_errors
